In [2]:
import open3d as o3d
import numpy as np
import cvxpy as cp
from scipy.special import binom
from scipy.integrate import quad
from scipy.optimize import newton
import matplotlib.pyplot as plt



Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [ ]:


mesh = o3d.io.read_triangle_mesh("plane_segments\Hyper_meshed_noise.stl")
mesh.compute_vertex_normals()
mesh.remove_duplicated_vertices()
edges = mesh.get_non_manifold_edges(allow_boundary_edges=False)

vertices=np.asarray(mesh.vertices)
raw_point_vertices= vertices[np.asarray(edges)]
points=np.unique(np.vstack(raw_point_vertices), axis=0)
point_cloud = o3d.geometry.PointCloud()
point_cloud.points = o3d.utility.Vector3dVector(points[250:550])
point_cloud.paint_uniform_color([1, 0, 0])  # Red
#o3d.visualization.draw_geometries([mesh, point_cloud])
#print(points[50:100])


## Edge extraction, corner detection, top edge segments

In [6]:
#%%timeit -n 10 -r 5

def order_perimeter(points, clockwise=True):
    centroid=np.mean(points,axis=0)
    angles=np.arctan2(points[:,1]-centroid[1], points[:, 0]-centroid[0])
    sort_order = np.argsort(angles)
    if clockwise:
        sort_order = sort_order[::-1] 
    #return points[sort_order]
    return points[sort_order], sort_order

def dot_product(points):
    forward_vecs=np.roll(points,shift=1, axis=0)-points
    backward_vecs=np.roll(points,shift=-1,axis=0)-points
    forward_vecs /= np.linalg.norm(forward_vecs, axis=1, keepdims=True)
    backward_vecs /= np.linalg.norm(backward_vecs, axis=1, keepdims=True)
    #einstine summation notation, more efficient than np.sum(f*b)
    dot_products=np.einsum("ij,ij->i",forward_vecs,backward_vecs)

    return dot_products

def split_perimeter(ordered_points, corner_indices):
    corner_indices = np.sort(corner_indices)  # Ensure indices are sorted
    segments=[]
    for i in range(len(corner_indices)):
        start_idx=corner_indices[i]
        end_idx=corner_indices[(i+1) % len(corner_indices)]
        if start_idx<end_idx:
            segment= ordered_points[start_idx: end_idx+1]
        else:
            segment=np.vstack([ordered_points[start_idx:],ordered_points[:end_idx+1]])
        segments.append(segment)
    return segments

def curvilinear_distance(segment):
    diffs=np.diff(segment,axis=0)
    distance=np.linalg.norm(diffs,axis=1)
    return(np.sum(distance))
    
mesh=o3d.io.read_triangle_mesh("plane_segments\Hyper_meshed_noise.stl")
#mesh=o3d.io.read_triangle_mesh("plane_segments\Curvy.stl")
#mesh=o3d.io.read_triangle_mesh(r"plane_segments\Non-planar.stl")
#mesh=o3d.io.read_triangle_mesh("plane_segments\Concave.stl")
mesh.compute_vertex_normals()
mesh.remove_duplicated_vertices()
edges_trial = mesh.get_non_manifold_edges(allow_boundary_edges=False)


edge_segments=np.asarray(mesh.vertices)[edges_trial]
edge_points=np.unique(edge_segments.reshape(-1,3), axis=0)

Proj_Eigs=np.array([[ 2.52179383e-07,  9.99999654e-01],
                    [ 9.99999622e-01, -9.74752318e-07],
                    [-8.69069051e-04, -8.31432961e-04]])

proj_points=edge_points@Proj_Eigs
ordered2D, clockwise_order=order_perimeter(proj_points)
ordered=edge_points[clockwise_order]
dps=dot_product(ordered2D)
#ordered=order_perimeter(edge_points)
#dps=dot_product(ordered)

#mask = np.abs(dps) < threshold
#theshold_dps=ordered[mask]
min_indices = np.argsort(np.abs(dps))[:4]
#print("Indices with lowest absolute values:", min_indices)
split=split_perimeter(ordered, min_indices)

segment_len=np.array([curvilinear_distance(segment) for segment in split])
top_two_indices = np.argsort(segment_len)[-2:][::-1]
top_two_curves = [split[top_two_indices[0]], split[top_two_indices[1]][::-1]]


# point_cloud = o3d.geometry.PointCloud()
# point_cloud.points = o3d.utility.Vector3dVector(np.vstack(top_two_curves))
# point_cloud.paint_uniform_color([1, 0, 0])  # Red
# o3d.visualization.draw_geometries([mesh, point_cloud])


## Adding Bézier functionality

In [7]:
#%%timeit -n 50 -r 5

def control_points_3D(curve):
    n=6
    Bpatch=np.zeros([n+1,n+1])
    for i in range(n+1):
        for j in range(n+1):
            Bpatch[i, j]=(-1)**(j-i)*binom(n,j)*binom(j,i)

    Pix=cp.Variable(n+1)
    Piy=cp.Variable(n+1)
    Piz=cp.Variable(n+1)

    n_opt=10
    projection=[i/n_opt for i in range(n_opt+1)]
    cost=0

    #TODO stop calling x/y/zvals so much and just use rows of curve!
    control_points=np.zeros([n_opt+1,3])
    for i in range(n_opt+1):
        if i==n_opt:
            control_points[i]=curve[-1]
        else:
            index=(i*len(curve))//n_opt
            control_points[i]=curve[index,:]

    Kx=Bpatch.T@Pix
    Ky=Bpatch.T@Piy
    Kz=Bpatch.T@Piz

    for i in range(n_opt+1):
        v=projection[i]
        V=np.array([v**k for k in range(n+1)])
        rsx=V@Kx
        rsy=V@Ky
        rsz=V@Kz
        cost += cp.square(cp.norm(cp.hstack([rsx - control_points[i][0], 
                                        rsy - control_points[i][1], 
                                        rsz - control_points[i][2]])))
    prob=cp.Problem(cp.Minimize(cost))
    prob.solve()

    Pi = np.column_stack([Pix.value, Piy.value, Piz.value])
    t_fine = np.linspace(0, 1, 100)
    bezier_points = np.array([bezier_curve3N(n, t, Pi) for t in t_fine])

    return bezier_points, Pi

def bezier_curve3N(n, t, control_points):

    if not (0 <= t <= 1):
        raise ValueError("t must be in the range (0,1)")

    if control_points.shape != (n + 1, 3):
        raise ValueError("control_points must be an (n+1) x 3 array")

    point = np.zeros(3)
    for i in range(n + 1):
        bernstein = binom(n, i) * (1 - t) ** (n - i) * t ** i
        point += bernstein * control_points[i]

    return point

bezier_points, Pi = control_points_3D(top_two_curves[1])

# point_cloud = o3d.geometry.PointCloud()
# point_cloud.points = o3d.utility.Vector3dVector(np.vstack(top_two_curves[1]))
# point_cloud.paint_uniform_color([1, 0, 0])  # Red
# bezier_fit=o3d.geometry.PointCloud()
# bezier_fit.points=o3d.utility.Vector3dVector(np.vstack(bezier_points))
# o3d.visualization.draw_geometries([mesh, bezier_fit, point_cloud])


## Evenly-spaced Sampler

In [8]:
# def bisection(n,control_points,target_length, t_m1, t_high, tol=1e-5):
#     t_origin=t_m1
#     t_low=t_m1
#     while t_high - t_low > tol:
#         t_mid = (t_low + t_high) / 2
#         arc_length_mid = bezier_arc_length(n, t_origin, t_mid, control_points)
#         #print(bezier_arc_length(n,0,t_mid,Pi))
#         #arc_length_mid=np.linalg.norm(bezier_curve3N(6,t_low,Pi)-bezier_curve3N(6,t_mid,Pi))
#         if arc_length_mid < target_length:
#             t_low = t_mid 
#         else:
#             t_high = t_mid

#     #return (t_low + t_high) / 2  # Approximate best t
#     return t_mid


##############################################################
def find_t_bisection(n, control_points, target_length, tol=1e-5):
    t_low, t_high = 0, 1  # Bezier curves are defined in [0,1]
    cumulative_t=[0]

    while t_low<=.98:
        new_t=bisection(n,control_points,target_length,t_low,t_high)
        cumulative_t.append(new_t)
        if len(cumulative_t)==2:
            t_conservative=3*(cumulative_t[1]-cumulative_t[0])
        t_low=new_t
        t_high=min(1,t_conservative+t_low)
    return cumulative_t


###################################################################

def bezier_derivative(n, t, control_points):
    """ Compute P'(t) for a Bezier curve. """
    derivative = np.zeros(3)
    for i in range(n):
        bernstein = n * binom(n-1, i) * (1-t)**(n-1-i) * t**i
        derivative += bernstein * (control_points[i+1] - control_points[i])
    return derivative

def arc_length_integrand(t, n, control_points):
    """ Returns ||P'(t)|| for integration. """
    return np.linalg.norm(bezier_derivative(n, t, control_points))

def bezier_arc_length(n, t_start, t_end, control_points):
    """ Compute arc length S(t) from t_start to t_end using numerical integration. """
    return quad(arc_length_integrand, t_start, t_end, args=(n, control_points))[0]

def find_t_newton(n, control_points, target_length, tol=1e-5):
    cumulative_t=[]
    t_conservative=0.2
    t_min=0
    while t_min<=1:
        cumulative_t.append(t_min)
        def f(t): return bezier_arc_length(n, t_min, t, control_points) - target_length
        def f_prime(t): return np.linalg.norm(bezier_derivative(n,t,control_points))
        if len(cumulative_t)==2:
            t_conservative=cumulative_t[1]-cumulative_t[0]
        t_guess=min(1,t_conservative+t_min)
        t_new = newton(f, t_guess, f_prime, tol=tol)
        t_min=t_new
    cumulative_t.append(1)    
        
    return cumulative_t

n=6
target_length=10

#t_value=find_t_bisection(n, Pi,target_length)
#%timeit t_value=find_t_bisection(n, Pi,target_length)
#evaluated=[bezier_arc_length(n,0,t,Pi) for t in t_value]
#print(evaluated)

t_value1=find_t_newton(n, Pi,target_length)
evaluated1=[bezier_arc_length(n,0,t,Pi) for t in t_value1]
even_spaced=[bezier_curve3N(n,t,Pi) for t in t_value1]
print(evaluated1)
#%timeit t_value1=find_t_newton(n, Pi,target_length)

point_cloud = o3d.geometry.PointCloud()
point_cloud.points = o3d.utility.Vector3dVector(even_spaced)
point_cloud.paint_uniform_color([1, 0, 0])  # Red
bezier_fit=o3d.geometry.PointCloud()
bezier_fit.points=o3d.utility.Vector3dVector(np.vstack(bezier_points))
o3d.visualization.draw_geometries([mesh, point_cloud])


[0.0, 9.99999999999868, 19.99999999999842, 29.99999999999889, 39.999999999999105, 49.99999999999901, 59.999999999998764, 69.99999999999854, 79.99999999999847, 89.99999999999864, 99.99999999999898, 109.99999999999922, 119.99999999999939, 129.9999999999994, 139.99999999999932, 149.99999999999912, 159.9999999999988, 169.9999999999986, 179.9999999999986, 189.999999999999, 199.99999999999966, 210.0000000000004, 220.0000000000008, 230.00000000000088, 240.000000000001, 250.00000000000108, 260.00000000000136, 270.00000000000136, 280.0000000000009, 290.00000000000034, 300.00000000000017, 310.0000000000001, 319.9999999999999, 329.9999999999993, 339.999999999999, 349.9999999999993, 360.00000000000006, 370.0000000000006, 380.00000000000074, 390.00000000000057, 399.9999999999999, 404.87229554554074]


## Edge Interpolation

In [9]:
#%%timeit

_, edge1_CP = control_points_3D(top_two_curves[0])
_, edge2_CP = control_points_3D(top_two_curves[1])
color_one=np.asarray([0, 1, 0])
color_two=np.asarray([0, 0, 1])
tot_passes=4
Control_Point_sets=[]
color=[]

for i in range(tot_passes):
        Control_Point_sets.append(edge1_CP*(1-(i)/(tot_passes-1))+edge2_CP*((i)/(tot_passes-1)))
        color.append(color_one*(1-(i)/(tot_passes-1))+color_two*((i)/(tot_passes-1)))

passes=[[bezier_curve3N(n,t,Pi) for t in find_t_newton(n, Pi,target_length)] for Pi in Control_Point_sets]
colors=np.vstack([col*np.ones((len(pas), 1)) for col, pas in zip(color,passes)])
passes=np.vstack(passes)


trial=o3d.geometry.PointCloud()
trial.points=o3d.utility.Vector3dVector(passes)
trial.colors=o3d.utility.Vector3dVector(colors)
o3d.visualization.draw_geometries([mesh, trial],mesh_show_back_face=True)
        


